# Titanic Disaster: Exploratory Data Analysis (EDA)

**Core Objective:**
The goal of this exploratory data analysis is to uncover patterns and identify which demographic and socioeconomic factors increased a passenger's likelihood of survival.

### Key Questions to Investigate:
- What was the overall survival rate among passengers?
- Did passenger gender affect survival odds ("women and children first")?
- How did socioeconomic standing (ticket class, fare) influence rescue priority?
- What role did age and family composition play in survival outcomes?

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/train.csv")

### 1. Initial Data Inspection & Health Check
In this section, we examine the structural dimensions, column data types, presence of duplicate records, and check for high-level data consistency.

In [3]:
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [4]:
print(f"(rows, columns) = {df.shape}")

(rows, columns) = (891, 12)


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df["PassengerId"].is_unique

True

### 2. Missing Values Assessment
Quantifying missing values per column with absolute counts and percentages to determine appropriate imputation or drop strategies.

In [8]:
missing_count = df.isnull().sum()
missing_pct = (missing_count / len(df)) * 100
missing_df = pd.DataFrame(
    {"Missing_Count": missing_count, "Percentage (%)": missing_pct.round(2)}
)
missing_df = (
    missing_df[missing_df["Missing_Count"] > 0]
    .sort_values(by="Missing_Count", ascending=False)
)
missing_df

,Missing_Count,Percentage (%)
Cabin,687,77.10
Age,177,19.87
Embarked,2,0.22


### 3. Summary Statistics
Statistical breakdown of numerical variables (identifying distributions and potential outliers) alongside categorical distributions.

In [9]:
print(df.describe().T)

print(df.describe(exclude=["number"]).T)

             count        mean         std   min       25%       50%    75%  \
PassengerId  891.0  446.000000  257.353842  1.00  223.5000  446.0000  668.5   
Survived     891.0    0.383838    0.486592  0.00    0.0000    0.0000    1.0   
Pclass       891.0    2.308642    0.836071  1.00    2.0000    3.0000    3.0   
Age          714.0   29.699118   14.526497  0.42   20.1250   28.0000   38.0   
SibSp        891.0    0.523008    1.102743  0.00    0.0000    0.0000    1.0   
Parch        891.0    0.381594    0.806057  0.00    0.0000    0.0000    0.0   
Fare         891.0   32.204208   49.693429  0.00    7.9104   14.4542   31.0   

                  max  
PassengerId  891.0000  
Survived       1.0000  
Pclass         3.0000  
Age           80.0000  
SibSp          8.0000  
Parch          6.0000  
Fare         512.3292  
         count unique                      top freq
Name       891    891  Braund, Mr. Owen Harris    1
Sex        891      2                     male  577
Ticket     891    6

### 4. Data Cleaning & Missing Value Imputation
In this phase, we address data anomalies and missing values in `Embarked`, `Age`, and `Cabin`. We preserve raw data integrity by performing all cleaning procedures on an explicit copy of the dataset (`df_clean`).

In [10]:
df_clean = df.copy()

In [11]:
embarked_mode = df_clean["Embarked"].mode()[0]
embarked_mode

'S'

In [12]:
df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode)

In [13]:
df_clean['Embarked'].isnull().sum()

np.int64(0)

### Handling Missing Values: `Age`
Instead of using a simple global mean, missing age values are imputed using the median age grouped by passenger class (`Pclass`) and gender (`Sex`). This preserves realistic socio-demographic age distributions.

In [14]:
age_medians = df_clean.groupby(["Pclass", "Sex"])["Age"].transform("median")

In [15]:
df_clean["Age"] = df_clean["Age"].fillna(age_medians)

In [16]:
df_clean['Age'].isnull().sum()

np.int64(0)

### Handling Missing Values: `Cabin`
With over 77% missing values, direct categorical encoding of individual cabins is impractical. Instead, we engineer two informative features:
1. `Has_Cabin`: A binary flag (1 if a cabin was assigned, 0 otherwise).
2. `Deck`: The deck level extracted from the initial letter of the cabin identifier ('Unknown' for missing records).

In [17]:
df_clean["Has_Cabin"] = df_clean["Cabin"].notnull().astype(int)

In [18]:
df_clean["Deck"] = df_clean["Cabin"].str[0].fillna("Unknown")

In [19]:
df_clean["Deck"].value_counts()

Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

In [20]:
df_clean.groupby("Has_Cabin")["Survived"].mean().round(3)

Has_Cabin
0    0.300
1    0.667
Name: Survived, dtype: float64

In [21]:
df_clean.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
Has_Cabin        0
Deck             0
dtype: int64

### Dropping Raw Cabin Column
Since the actionable signals have been extracted into `Has_Cabin` and `Deck`, the original sparse `Cabin` column is dropped to maintain a tidy DataFrame.

In [22]:
df_clean = df_clean.drop(columns=["Cabin"])

In [23]:
len(df_clean.columns)

13

In [24]:
df_clean.columns.tolist()

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Embarked',
 'Has_Cabin',
 'Deck']

### 5. Outlier & Anomaly Detection: `Fare`
We examine the distribution of fares, identify zero-fare passengers, and analyze extreme high-fare records using Interquartile Range (IQR) analysis to avoid improper data removal.

In [25]:
zero_fares = df_clean[df_clean["Fare"] == 0]
print(f"Total passengers with Fare = 0: {len(zero_fares)}")

Total passengers with Fare = 0: 15


In [26]:
zero_fares[["PassengerId", "Pclass", "Name", "Sex", "Age", "Ticket"]].head(5)

,PassengerId,Pclass,Name,Sex,Age,Ticket
179,180,3,"Leonard, Mr. Lionel",male,36.0,LINE
263,264,1,"Harrison, Mr. William",male,40.0,112059
271,272,3,"Tornquist, Mr. William Henry",male,25.0,LINE
277,278,2,"Parkes, Mr. Francis ""Frank""",male,30.0,239853
302,303,3,"Johnson, Mr. William Cahoone Jr",male,19.0,LINE


In [27]:
Q1 = df_clean["Fare"].quantile(0.25)
Q3 = df_clean["Fare"].quantile(0.75)
IQR = Q3 - Q1
upper_threshold = Q3 + (1.5 * IQR)

In [28]:
outliers_count = (df_clean["Fare"] > upper_threshold).sum()

In [29]:
print(f"Fare IQR: {IQR:.2f}")
print(f"Upper outlier threshold (Q3 + 1.5*IQR): {upper_threshold:.2f}")
print(f"Number of high-fare outliers: {outliers_count} ({outliers_count / len(df_clean) * 100:.2f}%)")

Fare IQR: 23.09
Upper outlier threshold (Q3 + 1.5*IQR): 65.63
Number of high-fare outliers: 116 (13.02%)


In [30]:
print("Top 5 Highest Fare Records:")
display(
    df_clean.sort_values(by="Fare", ascending=False)[
        ["PassengerId", "Name", "Pclass", "Fare", "Survived"]
    ].head(5)
)

Top 5 Highest Fare Records:


,PassengerId,Name,Pclass,Fare,Survived
679,680,"Cardeza, Mr. Thomas Drake Martinez",1,512.3292,1
258,259,"Ward, Miss. Anna",1,512.3292,1
737,738,"Lesurer, Mr. Gustave J",1,512.3292,1
88,89,"Fortune, Miss. Mabel Helen",1,263.0000,1
438,439,"Fortune, Mr. Mark",1,263.0000,0


### Outlier & Anomaly Detection: `Age`
Validating boundary values for age (infants and elderly passengers) to confirm historical plausibility.

In [31]:
min_age_passengers = df_clean[df_clean["Age"] < 1]
max_age_passengers = df_clean[df_clean["Age"] >= 70]

In [32]:
print(f"Infants under 1 year old: {len(min_age_passengers)}")
print(f"Elderly passengers aged 70 or above: {len(max_age_passengers)}")

Infants under 1 year old: 7
Elderly passengers aged 70 or above: 7


In [33]:
max_age_passengers[["PassengerId", "Name", "Age", "Pclass", "Survived"]]

,PassengerId,Name,Age,Pclass,Survived
96,97,"Goldschmidt, Mr. George B",71.0,1,0
116,117,"Connors, Mr. Patrick",70.5,3,0
493,494,"Artagaveytia, Mr. Ramon",71.0,1,0
630,631,"Barkworth, Mr. Algernon Henry Wilson",80.0,1,1
672,673,"Mitchell, Mr. Henry Michael",70.0,2,0
745,746,"Crosby, Capt. Edward Gifford",70.0,1,0
851,852,"Svensson, Mr. Johan",74.0,3,0
